In [40]:
import numpy as np


class Variable:
    def __init__(self, data):
        if data is not None:
            if not isinstance(data, np.ndarray):
                raise TypeError('{} is not supported'.format(type(data)))

        self.data = data
        self.grad = None
        self.creator = None
        self.generation = 0

    def set_creator(self, f):
        self.creator = f
        self.generation = f.generation + 1

    def backward(self):
        if self.grad is None:
            self.grad = np.ones_like(self.data)
        funcs = []
        funcs.append(self.creator)
        seen_set = set()
        def add_func(f):
            if f not in seen_set:
                funcs.append(f)
                seen_set.add(f)
                funcs.sort(key=lambda x: x.generation)
        #add_func(self.creator)
        while funcs:
            f = funcs.pop()
            # ys = f.outputs
            gys = [y.grad for y in f.outputs]
            # xs = f.inputs
            # x.grad = f.backward(y.grad)
            gxs = f.backward(*gys)
            for x, gx in zip(f.inputs, gxs):
                if x.grad:
                    x.grad = x.grad + gx
                else:
                    x.grad = gx
                if x.creator is not None:
                    funcs.append(self.creator)
    def cleargrad(self):
        self.grad = None
class Function:
    def __call__(self, *inputs):
        # x = input.data # unzip
        xs = [input.data for input in inputs]
        ys = self.forward(*xs)
        if not isinstance(ys, tuple):
            ys = (ys,)
        outputs = [Variable(as_array(y)) for y in ys]
        self.inputs = inputs
        self.outputs = outputs
        self.generation = np.max([input.generation for input in inputs])
        for output in outputs:
            output.set_creator(self)
        return outputs if len(outputs) > 1 else outputs[0]

    def forward(self, x):
        raise NotImplementedError

    def backward(self, gy):
        raise NotImplementedError


def as_array(x):
    if np.isscalar(x):
        return np.array(x)
    return x


class Square(Function):
    def forward(self, x):
        return x ** 2

    def backward(self, gy):
        x = self.inputs[0].data
        gx = 2 * x * gy
        return gx


class Exp(Function):
    def forward(self, x):
        return np.exp(x)

    def backward(self, gy):
        x = self.inputs[0].data
        gx = np.exp(x) * gy
        return gx


class Add(Function):
    def forward(self, x0, x1):
        y = x0 + x1
        return y

    def backward(self, gy):
        return gy, gy


def exp(x):
    return Exp()(x)


def square(x):
    return Square()(x)

def add(x0, x1):
    return Add()(x0,x1)

In [41]:
x = Variable(np.array(2.0))
a = square(x)
y = add(square(a), square(a))
y.backward()
print(y.data)
print(x.grad)

KeyboardInterrupt: 

In [2]:
x = Variable(np.array([0.5]))
y = Variable(np.array([21.5]))
f = Add()
z = f(x,y)
print(z.data)

[22.]


In [3]:
A = Square()
B = Exp()
C = Square()

x = Variable(np.array([0.5,1,1.5,2]))
a = A(x)
print(a.creator)
b = B(a)
print(b.creator)
y = C(b)
print(y.creator)

y.backward()
print(y.grad)
print(b.grad)
print(a.grad)
print(x.grad)



AttributeError: 'tuple' object has no attribute 'data'

In [43]:
np.exp(0.25) *2 * np.exp(0.25)

np.float64(3.297442541400256)

In [14]:
def numerical_diff(f, x, eps=1e-3):
    x0, x1 = Variable(x.data - eps), Variable(x.data + eps)
    y0, y1 = f(x0), f(x1)
    return (y1.data - y0.data) / ( 2 * eps)

In [21]:
x = Variable(np.array([0]))
f = Square()
numerical_diff(f, x, eps=10)

array([0.])